Problem Statement: Analysis of Student Dropout Determinants in Kenyan Higher Education

1. Background & Context
Student dropout from higher education institutions is a critical challenge in Kenya, leading to significant individual, social, and economic costs. These include wasted educational resources, reduced skilled workforce, increased unemployment, and lower lifetime earnings for the individuals who drop out. Understanding the multifaceted factors that contribute to this phenomenon is essential for developing effective, data-driven retention strategies. This dataset captures information on 13,000+ students, providing a rich opportunity to explore the complex interplay of academic, financial, demographic, and engagement-related factors.

2. Problem Identification
The core problem is the unexplained and unmitigated student dropout phenomenon in the Kenyan higher education context. While some factors are intuitively understood (e.g., low grades), the relative importance and interaction of various predictors—such as financial stress (tuition vs. loans), academic preparation (High School Grade vs. University GPA), program characteristics (type, level), and student engagement (attendance, extracurricular participation)—are not clearly defined for this specific population.
Specifically, the dataset allows us to investigate key questions that are currently unanswered:
•What is the primary driver of dropout: pre-university academic preparedness (High School Grade), current academic performance (University GPA), or financial constraints (tuition fees, loan availability)?
•How do financial factors (e.g., Tuition_Fees, Student_Loan amount, Scholarship status) interact with academic factors to influence dropout risk?
•Does the impact of these factors vary significantly by student demographics (e.g., Gender, County), the program of study (Program, Program_Level), or levels of engagement (Attendance_Rate, Extracurricular_Participation)?
•Can a predictive model be built to identify students at high risk of dropping out before it happens, based on their first-year or pre-enrollment data?

3. Data Suitability
The provided dataset is well-structured for addressing this problem. It contains:
•Target Variable: A clear binary outcome variable (Dropout), where 1 indicates a student dropped out and 0 indicates they persisted.
•Predictor Variables: A wide range of potential predictors, including:
•Academic: High_School_Grade, University_GPA
•Financial: Scholarship (binary), Tuition_Fees, Student_Loan
•Demographic: Age, Gender, County
•Institutional: Program (e.g., Law, Engineering), Program_Level (e.g., Undergraduate, Postgraduate)
•Engagement: Attendance_Rate (%), Extracurricular_Participation (binary)
•Family Background: Parental_Education_Level
•Scale & Completeness: With over 13,000 records, the dataset has sufficient statistical power to detect meaningful patterns. However, a preliminary check for missing or inconsistent data is necessary before analysis.

4. Objectives of the Analysis
The primary objective is to identify, quantify, and model the key determinants of student dropout in this Kenyan higher education dataset. Specific sub-objectives include:
1.Descriptive Analysis: To profile the student population and compare the characteristics of those who dropped out versus those who persisted, across all academic, financial, demographic, and engagement variables.
2.Exploratory Factor Analysis: To investigate the relationship between independent variables (e.g., Tuition Fees and Student Loan) and the dependent variable (Dropout), and to explore correlations between predictors (e.g., High_School_Grade and University_GPA).
3.Predictive Modeling: To develop and evaluate a classification model (e.g., Logistic Regression, Random Forest, XGBoost) that can accurately predict a student's risk of dropping out based on available data. Feature importance from this model will reveal the most influential factors.
4.Actionable Insights Generation: To translate the findings from the analysis and model into clear, actionable recommendations for university administrators and policymakers. This includes identifying which student segments are most at risk and suggesting targeted interventions (e.g., financial aid restructuring, academic support programs, or engagement initiatives).

5. Expected Outcomes & Impact
A successful analysis will produce:
•A ranked list of the most significant predictors of student dropout in this context.
•A validated predictive model for early identification of at-risk students.
•Data-driven policy recommendations to reduce dropout rates, improve student retention, and optimize the allocation of institutional resources (e.g., targeting financial aid or tutoring to those who need it most).
Ultimately, this analysis aims to contribute to a more efficient and equitable higher education system in Kenya, ensuring that more students who enroll are able to successfully complete their chosen programs.

In [ ]:
# Importing the necessary libraries for the project

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Loading the dataset
School_df = pd.read_csv("student.csv")
School_df.head()

In [ ]:
School_df.info()

In [ ]:
School_df.isna().sum()

# Assumptions we've made
1. Age - 16yrs and above is eligible to join an Undergrad program
2. 

### Creating some new features in the df

In [ ]:
# Creating the Age column

# convert to datetime
School_df["Birthdate"] = pd.to_datetime(School_df["Birthdate"], errors="coerce")

# today's date
today = datetime.today()

# create age column
School_df["Age"] = School_df["Birthdate"].apply(
    lambda x: today.year - x.year - ((today.month, today.day) < (x.month, x.day))
    if pd.notnull(x) else None
)

In [ ]:
columns_to_drop = ['BURSARYBATCHNUMBER', 'Unnamed: 21', 'PHYSICALLYCHALLENGED','Loan_serial_number','InstitutionCode', 'Adm No','COURSECODE','INDEXNUMBER','BursaryAllocated','applicanttype', 'Birthdate']

# Use axis=1 to tell pandas these are column names, not row indexes
School_df = School_df.drop(columns_to_drop, axis=1)

School_df.info()

In [ ]:
School_df.isna().sum()

In [ ]:
#After careful evaluation, we decided to drop the rows in the columns with missing values

School_df=School_df.dropna(subset=["Gender","County","ProgramCost","CourseCategory",
                                   "Father_educ_level","Mother_educ_level","LoanStatus", "Age"])
School_df.info()

In [ ]:
# 
School_df[['UniversityType', 'Sponsored']] = School_df['Category'].str.extract(
    r'^(Public|Private).*?(GovtSponsored|SelfSponsored)$'
)
School_df.head(10)

In [ ]:
School_df = School_df[(School_df['LoanproductCode'] != 'DL6') & 
                      (School_df['LoanproductCode'] != 'VC')]

In [ ]:
School_df.shape

In [ ]:
School_df.info()

In [ ]:
plt.figure(figsize=(6,4))

sns.boxplot(y=School_df['Age'])

plt.title('Age Distribution')
plt.show()

# We need to use data from 17 yrs to 22 or 23, that's the data we have, we do not have sufficient data on the rest

In [ ]:
School_df= School_df[(School_df['Age'] >= 15) & (School_df['Age'] <= 50)] 

In [ ]:
School_df.Age.value_counts()

In [ ]:
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.facecolor": "#FAFAFA",
    "axes.facecolor":   "#FAFAFA",
    "axes.spines.top":  False,
    "axes.spines.right": False,
    "font.family":      "DejaVu Sans",
})
 
# Colour palette
BLUE   = "#3266AD"
RED    = "#E24B4A"
GREEN  = "#1D9E75"
AMBER  = "#EF9F27"
GRAY   = "#73726C"
PURPLE = "#534AB7"
PAL    = [BLUE, RED]

In [ ]:
plt.figure(figsize=(6,4))

sns.boxplot(y=School_df['Age'])

plt.title('Age Distribution')
plt.show()

In [ ]:
# School_df.groupby('County')['Drop_out'].mean().sort_values(ascending=False)
# top_counties = (
#     School_df.groupby('County')['Drop_out']
#     .mean()
#     .sort_values(ascending=False)
#     .head(7)
#     .reset_index()
# )
# plt.figure(figsize=(10,6))

# sns.barplot(data=top_counties, x='Drop_out', y='County', palette='Reds_r')

# plt.title('Top 7 Counties by Dropout Rate')
# plt.xlabel('Dropout Rate')
# plt.ylabel('County')

# plt.show()

In [ ]:
bins = [0, 17, 20, 24, 30, 34, 39, 100]
labels = ['<18', '18-20', '21-24', '25-30', '30-35','36-40','40-45']

School_df['age_group'] = pd.cut(School_df['Age'], bins=bins, labels=labels)    

sns.countplot(x='age_group', hue='Drop_out', data=School_df)

plt.title('Age Group vs Dropout')
plt.show()

In [ ]:
corr = School_df.corr()

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")

plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# School_df.to_csv('cleaned.csv')

In [ ]:
plt.figure(figsize=(12, 7))
df_ratio = School_df.dropna(subset=['ProgramCost', 'TotalLoanAllocated']).copy()
df_ratio = df_ratio[df_ratio['ProgramCost'] > 0]
df_ratio['LoanToCostRatio'] = df_ratio['TotalLoanAllocated'] / df_ratio['ProgramCost']
# Cap ratio at 2 for better visualization
df_ratio['LoanToCostRatio'] = df_ratio['LoanToCostRatio'].clip(upper=2)

box = sns.boxplot(x='UniversityType', y='LoanToCostRatio', data=df_ratio, palette='Set1')
plt.title('Loan to Program Cost Ratio by University Type', fontsize=16, fontweight='bold')
plt.xlabel('University Type', fontsize=12)
plt.ylabel('Loan to Cost Ratio', fontsize=12)
plt.axhline(y=1, color='red', linestyle='--', linewidth=2, label='Full Coverage (100%)')
plt.legend()

stats = df_ratio.groupby('UniversityType')['LoanToCostRatio'].agg(['mean', 'median'])
for i, univ_type in enumerate(stats.index):
    plt.text(i, stats.loc[univ_type, 'median'] + 0.05, 
             f'Median: {stats.loc[univ_type, "median"]:.2f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

This shows What portion of program costs are covered by loans (ratio = Loan Amount / Program Cost), comparing Public vs Private universities.
Public universities has higher ratio due to government support and lower costs compared to Private universities which has lower ratio due to higher fees

In [ ]:
plt.figure(figsize=(12, 7))
df_age_group = School_df.dropna(subset=['Age']).copy()
df_age_group = df_age_group[(df_age_group['Age'] >= 18) & (df_age_group['Age'] <= 60)]

# Create age groups
bins = [18, 22, 27, 32, 40, 100]
labels = ['18-22', '23-27', '28-32', '33-40', '40+']
df_age_group['AgeGroup'] = pd.cut(df_age_group['Age'], bins=bins, labels=labels, right=False)

# Calculate dropout rate by age group
dropout_by_age = df_age_group.groupby('AgeGroup')['Drop_out'].agg(['mean', 'count'])
dropout_by_age = dropout_by_age[dropout_by_age['count'] > 10]  # Filter small groups

bars = plt.bar(range(len(dropout_by_age)), dropout_by_age['mean'] * 100, 
               color='#e74c3c', edgecolor='black', alpha=0.7)
plt.xticks(range(len(dropout_by_age)), dropout_by_age.index)
plt.title('Dropout Rate by Age Group', fontsize=16, fontweight='bold')
plt.xlabel('Age Group', fontsize=12)
plt.ylabel('Dropout Rate (%)', fontsize=12)
plt.ylim(0, 100)

# Add percentage labels
for i, (bar, rate) in enumerate(zip(bars, dropout_by_age['mean'])):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{rate*100:.1f}%', 
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.axhline(y=df_age_group['Drop_out'].mean() * 100, color='blue', linestyle='--', 
            label=f'Overall Average: {df_age_group["Drop_out"].mean()*100:.1f}%')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

scholarship_counts = School_df['ScholarshipApplied'].value_counts()
labels = ['No Scholarship', 'Applied for Scholarship']
colors_pie = ['#95a5a6', '#2ecc71']
axes[0].pie(scholarship_counts.values, labels=labels, autopct='%1.1f%%', colors=colors_pie, 
            startangle=90, explode=(0.05, 0.05))
axes[0].set_title('Scholarship Application Proportion', fontsize=14, fontweight='bold')

# Box plot - Loan comparison
sns.boxplot(x='ScholarshipApplied', y='TotalLoanAllocated', data=School_df, ax=axes[1], palette=['#95a5a6', '#2ecc71'])
axes[1].set_title('Loan Amount: Scholarship Applicants vs Non-Applicants', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Scholarship Applied')
axes[1].set_ylabel('Total Loan Allocated (KES)')
axes[1].set_xticklabels(['No', 'Yes'])

# Add statistics
for i, val in enumerate([0, 1]):
    median_val = School_df[School_df['ScholarshipApplied'] == val]['TotalLoanAllocated'].median()
    axes[1].text(i, median_val + 5000, f'Median: {median_val:,.0f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 7))
year_loan = School_df.groupby('ExamYear')['TotalLoanAllocated'].agg(['mean', 'median', 'count'])
year_loan = year_loan[year_loan['count'] > 5]  # Filter years with few observations

plt.plot(year_loan.index, year_loan['mean'], 'o-', linewidth=2, markersize=8, label='Mean Loan', color='#2ecc71')
plt.plot(year_loan.index, year_loan['median'], 's--', linewidth=2, markersize=8, label='Median Loan', color='#3498db')
plt.fill_between(year_loan.index, year_loan['mean'] - year_loan['mean'].std(), 
                 year_loan['mean'] + year_loan['mean'].std(), alpha=0.2, color='#2ecc71')

plt.title('Loan Amount Trend by Exam Year', fontsize=16, fontweight='bold')
plt.xlabel('Exam Year', fontsize=12)
plt.ylabel('Loan Amount (KES)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



In [ ]:
plt.figure(figsize=(10, 6))
status_counts = School_df['LoanStatus'].value_counts()
colors = ['#2ecc71', '#f39c12', '#e74c3c', '#95a5a6']
bars = plt.bar(status_counts.index, status_counts.values, color=colors[:len(status_counts)], edgecolor='black', linewidth=1.5)
plt.title('Distribution of Loan Status', fontsize=16, fontweight='bold')
plt.xlabel('Loan Status', fontsize=12)
plt.ylabel('Count', fontsize=12)
for bar, count in zip(bars, status_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(count), 
             ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
box = sns.boxplot(x='Gender', y='TotalLoanAllocated', data=School_df, palette='Set2')
plt.title('Loan Amount Distribution by Gender', fontsize=16, fontweight='bold')
plt.xlabel('Gender', fontsize=12)
plt.ylabel('Total Loan Allocated (KES)', fontsize=12)
# Add median values
medians = School_df.groupby('Gender')['TotalLoanAllocated'].median()
for i, gender in enumerate(medians.index):
    plt.text(i, medians[gender] + 5000, f'Median: {medians[gender]:,.0f}', 
             ha='center', fontsize=10)
plt.tight_layout()
plt.show()



In [ ]:
plt.figure(figsize=(10, 6))
box = sns.boxplot(x='Sponsored', y='TotalLoanAllocated', data=School_df, palette='Set3')
plt.title('Loan Amount by Sponsorship Type', fontsize=16, fontweight='bold')
plt.xlabel('Sponsorship Type', fontsize=12)
plt.ylabel('Total Loan Allocated (KES)', fontsize=12)
medians = School_df.groupby('Sponsored')['TotalLoanAllocated'].median()
for i, sponsor in enumerate(medians.index):
    plt.text(i, medians[sponsor] + 5000, f'Median: {medians[sponsor]:,.0f}', 
             ha='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
# Sample for better visualization (if too many points)
df_sample = School_df.dropna(subset=['ProgramCost', 'TotalLoanAllocated'])
if len(df_sample) > 2000:
    df_sample = df_sample.sample(n=2000, random_state=42)

scatter = sns.scatterplot(x='ProgramCost', y='TotalLoanAllocated', hue='Category', 
                          data=df_sample, alpha=0.6, s=50)

# Add trend line
z = np.polyfit(df_sample['ProgramCost'], df_sample['TotalLoanAllocated'], 1)
p = np.poly1d(z)
plt.plot(df_sample['ProgramCost'].sort_values(), p(df_sample['ProgramCost'].sort_values()), 
         "r--", linewidth=2, label=f'Trend: y={z[0]:.2f}x+{z[1]:.0f}')

plt.title('Program Cost vs Loan Allocated', fontsize=16, fontweight='bold')
plt.xlabel('Program Cost (KES)', fontsize=12)
plt.ylabel('Total Loan Allocated (KES)', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))
# Get top 10 course categories
top_courses = School_df['CourseCategory'].value_counts().head(10).index
df_top_courses = School_df[School_df['CourseCategory'].isin(top_courses)]

box = sns.boxplot(x='CourseCategory', y='TotalLoanAllocated', data=df_top_courses, palette='viridis')
plt.title('Loan Amount by Course Category (Top 10)', fontsize=16, fontweight='bold')
plt.xlabel('Course Category', fontsize=12)
plt.ylabel('Total Loan Allocated (KES)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()



In [ ]:
plt.figure(figsize=(12, 7))
df_age = School_df.dropna(subset=['Age', 'Gender'])
# Filter reasonable ages (18-60)
df_age = df_age[(df_age['Age'] >= 18) & (df_age['Age'] <= 60)]

for gender in df_age['Gender'].unique():
    subset = df_age[df_age['Gender'] == gender]
    sns.histplot(subset['Age'], kde=True, label=gender, alpha=0.5, bins=30)

plt.title('Age Distribution of Applicants by Gender', fontsize=16, fontweight='bold')
plt.xlabel('Age (years)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
plt.figure(figsize=(10, 6))
dropout_by_type = pd.crosstab(School_df['UniversityType'], School_df['Drop_out'])
dropout_by_type.columns = ['Not Dropped Out', 'Dropped Out']

dropout_by_type.plot(kind='bar', stacked=True, color=['#2ecc71', '#e74c3c'], edgecolor='black')
plt.title('Dropout Status by University Type', fontsize=16, fontweight='bold')
plt.xlabel('University Type', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.legend(title='Dropout Status')
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()



In [ ]:
plt.figure(figsize=(14, 8))
county_loan = School_df.groupby('County')['TotalLoanAllocated'].mean().sort_values(ascending=False).head(20)

bars = plt.barh(range(len(county_loan)), county_loan.values, color='skyblue', edgecolor='navy')
plt.title('Average Loan Amount by County (Top 20)', fontsize=16, fontweight='bold')
plt.xlabel('Average Loan Allocated (KES)', fontsize=12)
plt.ylabel('County', fontsize=12)
plt.yticks(range(len(county_loan)), county_loan.index)

# Add value labels
for i, (bar, val) in enumerate(zip(bars, county_loan.values)):
    plt.text(val + 5000, bar.get_y() + bar.get_height()/2, f'{val:,.0f}', 
             va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))
inst_loan = School_df.groupby('InstitutionName')['TotalLoanAllocated'].mean().sort_values(ascending=False).head(15)

bars = plt.barh(range(len(inst_loan)), inst_loan.values, color='lightcoral', edgecolor='darkred')
plt.title('Average Loan Amount by Institution (Top 15)', fontsize=16, fontweight='bold')
plt.xlabel('Average Loan Allocated (KES)', fontsize=12)
plt.ylabel('Institution', fontsize=12)
plt.yticks(range(len(inst_loan)), inst_loan.index, fontsize=9)



plt.tight_layout()
plt.show()



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Mother's education
mother_edu = School_df.groupby('Mother_educ_level')['TotalLoanAllocated'].mean().sort_values()
axes[0].barh(mother_edu.index, mother_edu.values, color='#3498db', edgecolor='black')
axes[0].set_title("Average Loan by Mother's Education Level", fontsize=14, fontweight='bold')
axes[0].set_xlabel('Average Loan Allocated (KES)')
axes[0].set_ylabel('Mother\'s Education Level')

# Father's education
father_edu = School_df.groupby('Father_educ_level')['TotalLoanAllocated'].mean().sort_values()
axes[1].barh(father_edu.index, father_edu.values, color='#e67e22', edgecolor='black')
axes[1].set_title("Average Loan by Father's Education Level", fontsize=14, fontweight='bold')
axes[1].set_xlabel('Average Loan Allocated (KES)')
axes[1].set_ylabel('Father\'s Education Level')

plt.tight_layout()
plt.show()

